# Session 12 · Ask my docs, with an eval

**From Prompt to Production · EvisionC**

1. Set **ROUTE** in the first cell to your model route from Session 2 (`claude`, `gemini`, `openai` or `openrouter`).
2. Add your API key in Colab Secrets (key icon, left sidebar, Notebook access on) under the name for your route: `ANTHROPIC_API_KEY`, `GEMINI_API_KEY`, `OPENAI_API_KEY` or `OPENROUTER_API_KEY`.
3. Run the cells from top to bottom (Shift+Enter). **Lumo track:** everything runs as is. **Own case:** the comments say what to swap.

The guided player explains every cell.

In [ ]:
# Cell 1 · setup (the same in every lab)
!git clone -q --depth 1 https://github.com/lrebelfor/lumo-sandbox /content/lumo 2>/dev/null || git -C /content/lumo pull -q
!pip install -q anthropic openai chromadb
import sys; sys.path.insert(0, "/content/lumo")
import lumo
from course_llm import LLM

ROUTE = "claude"   # your model route from Session 2: "claude", "openai", "gemini", "openrouter" (or "ollama" outside Colab)
llm = LLM(ROUTE)
print("Connected:", llm)
import chromadb, glob, os, re

# 1. YOUR DOCUMENTS. Lumo track: the handbook. Own case: set CASE = "own" and upload .txt/.md files
#    into a folder called "docs" (Colab: Files panel on the left).
CASE = "lumo"
if CASE == "lumo":
    documents = lumo.handbook()
else:
    documents = [(os.path.basename(p), open(p, encoding="utf-8", errors="ignore").read()) for p in glob.glob("docs/*")]

# 2. CHUNK each document by heading, keep the file name as the source
chunks = []
for name, text in documents:
    for n, part in enumerate(p.strip() for p in re.split(r"\n(?=#)", text) if p.strip()):
        chunks.append((f"{name}#{n}", part))
kb = chromadb.Client().get_or_create_collection("mydocs")
kb.add(ids=[i for i, _ in chunks], documents=[t for _, t in chunks])
print(len(chunks), "chunks from", len(documents), "documents")

SYSTEM = """Answer using ONLY the sources. Cite the source id in square brackets after each fact.
If the answer isn't in the sources, reply exactly: I don't know."""

def ask_my_docs(question, k=3):
    hits = kb.query(query_texts=[question], n_results=min(k, len(chunks)))
    sources = "\n\n".join(f"[{i}]\n{t}" for i, t in zip(hits["ids"][0], hits["documents"][0]))
    return llm.ask(f"<sources>\n{sources}\n</sources>\n\nQuestion: {question}", system=SYSTEM).text

print(ask_my_docs("Do I have to return a broken lamp?"))

In [ ]:
# 3. THE EVAL: 20 questions with the fact a correct answer must contain.
# Lumo track: the ready-made set (4 questions the handbook can't answer expect "I don't know").
# Own case: write your 20, including 3-4 your documents can't answer.
evals = lumo.eval_rag() if CASE == "lumo" else [
    ("Which countries do you ship to?", "Belgium"),
    # ... your questions here
]

def judge(answer, must_contain):
    """Simple and strict: the expected fact must appear in the answer (case-insensitive)."""
    return must_contain.lower() in answer.lower()

score = 0
for q, fact in evals:
    a = ask_my_docs(q)
    ok = judge(a, fact)
    score += ok
    print(("PASS" if ok else "FAIL"), "|", q, "\n      ", a.replace("\n", " ")[:160])
pct = 100 * score / len(evals)
print(f"\nScore: {score}/{len(evals)} = {pct:.0f}%  ->  Gate 1 needs 80%")